In [11]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!ls /content/drive/MyDrive

 archive.zip  'Colab Notebooks'   My_resume.pdf


In [13]:
import zipfile

zip_path = "/content/drive/MyDrive/archive.zip"
extract_path = "/content/afhq"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted!")

Extracted!


In [14]:
!ls /content/afhq

afhq


In [15]:
!ls /content/animal_dataset/afhq

train  val


In [16]:
train_dir = "/content/animal_dataset/afhq/train"
val_dir   = "/content/animal_dataset/afhq/val"

In [29]:
import tensorflow as tf

IMG_SIZE = (128,128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 14630 files belonging to 3 classes.
Found 1500 files belonging to 3 classes.
Classes: ['cat', 'dog', 'wild']


In [30]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [31]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [32]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

In [27]:
# base_model = tf.keras.applications.MobileNetV2(
#     input_shape=(128,128,3),
#     include_top=False,
#     weights="imagenet"
# )

# base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [33]:
inputs = tf.keras.Input(shape=(128,128,3))

x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)

x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

In [34]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,045,955 (11.62 MB)

 Trainable params: 787,971 (3.01 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [35]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 39s 63ms/step - accuracy: 0.9679 - loss: 0.0991 - val_accuracy: 0.9913 - val_loss: 0.0256
Epoch 2/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.9834 - loss: 0.0527 - val_accuracy: 0.9953 - val_loss: 0.0191
Epoch 3/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.9871 - loss: 0.0396 - val_accuracy: 0.9973 - val_loss: 0.0185
Epoch 4/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 42s 58ms/step - accuracy: 0.9878 - loss: 0.0370 - val_accuracy: 0.9933 - val_loss: 0.0246
Epoch 5/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 42s 60ms/step - accuracy: 0.9888 - loss: 0.0358 - val_accuracy: 0.9973 - val_loss: 0.0160
Epoch 6/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.9865 - loss: 0.0376 - val_accuracy: 0.9933 - val_loss: 0.0243
Epoch 7/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.9861 - loss: 0.0415 - val_accuracy: 0.9947 - val_loss: 0.0222
Epoch 8/10
458/458 ━━━━━━━━━━━━━━━━━━━━ 28s 60ms/step - accuracy: 0.9896 - loss: 0.0312 - 